In [1]:
# https://pypi.org/project/doi2pdf/
!pip install doi2pdf

In [2]:
import pandas as pd

hayek_refs = pd.read_csv("../data/raw/scopus_papers_citing_hayek.csv")
mises_refs = pd.read_csv("../data/raw/scopus_papers_citing_mises.csv")
dois = mises_refs['DOI'].dropna().astype(str).unique()


In [3]:
output_dir = "../data/processed/pdfs"

In [4]:
import re

def doi_to_filename(doi: str) -> str:
    """Convert a DOI to a Windows-friendly filename."""
    # Replace characters that are invalid in Windows filenames
    filename = re.sub(r'[\\/*?:"<>|]', '_', doi)
    # Replace forward slashes with underscores
    filename = filename.replace('/', '_')
    # Strip leading/trailing spaces and dots (Windows doesn't like these)
    filename = filename.strip('. ')
    return filename

# Example
doi = "10.1017/S0266267125000057"
print(doi_to_filename(doi))  # Output: 10.1017_S0266267125000057

10.1017_S0266267125000057


Primeira Onda de Coleta

In [5]:
import os
import csv
import re
import shutil
from concurrent.futures import ThreadPoolExecutor, as_completed
from threading import Lock
from doi2pdf import doi2pdf
from doi2pdf.main import NotFoundError
from requests.exceptions import ConnectionError as RequestsConnectionError
from tqdm import tqdm

N_THREADS = 20
processed_file = "../data/processed/processed_dois.csv"
csv_lock = Lock()
processed_lock = Lock()

# Load already processed DOIs
processed_dois = {}
if os.path.exists(processed_file):
    with open(processed_file, "r", newline="", encoding="utf-8") as f:
        for row in csv.DictReader(f):
            processed_dois[row["doi"]] = row

def get_file_extension(filepath: str) -> str:
    """Detect if file is HTML or PDF based on its magic bytes/header."""
    with open(filepath, "rb") as f:
        header = f.read(100)
    if header[:5] == b"%PDF-":
        return ".pdf"
    header_lower = header.lower()
    if b"<!doc" in header_lower or b"<html" in header_lower:
        return ".html"
    return ".bin"

def save_processed(doi: str, filepath: str, filetype: str):
    os.makedirs(os.path.dirname(processed_file), exist_ok=True)
    with csv_lock:
        file_exists = os.path.exists(processed_file)
        with open(processed_file, "a", newline="", encoding="utf-8") as f:
            writer = csv.DictWriter(f, fieldnames=["doi", "filepath", "filetype"])
            if not file_exists:
                writer.writeheader()
            writer.writerow({"doi": doi, "filepath": filepath, "filetype": filetype})

def doi_to_filename(doi: str) -> str:
    """Convert a DOI to a Windows-friendly filename."""
    filename = re.sub(r'[\\/*?:"<>|]', '_', doi)
    filename = filename.replace('/', '_')
    filename = filename.strip('. ')
    return filename

def process_doi(doi: str, pbar: tqdm) -> None:
    with processed_lock:
        if doi in processed_dois:
            tqdm.write(f"  ⏭ Skipping (already processed): {doi}")
            pbar.update(1)
            return

    tqdm.write(doi)
    try:
        temp_path = f"{output_dir}/{doi_to_filename(doi)}"
        doi2pdf(doi, output=temp_path)

        ext = get_file_extension(temp_path)
        final_path = temp_path + ext
        shutil.move(temp_path, final_path)

        if ext != ".pdf":
            tqdm.write(f"  ⚠ File is not a PDF ({ext}): {final_path}")

        save_processed(doi, final_path, ext.lstrip("."))
        with processed_lock:
            processed_dois[doi] = {"doi": doi, "filepath": final_path, "filetype": ext.lstrip(".")}

    except NotFoundError:
        tqdm.write(f"  ✗ PDF not found for DOI: {doi}")
        save_processed(doi, "", "not_found")
        with processed_lock:
            processed_dois[doi] = {"doi": doi, "filepath": "", "filetype": "not_found"}

    except RequestsConnectionError as e:
        tqdm.write(f"  ✗ Connection error for DOI {doi}: {e}")
        save_processed(doi, "", "connection_error")
        with processed_lock:
            processed_dois[doi] = {"doi": doi, "filepath": "", "filetype": "connection_error"}

    finally:
        pbar.update(1)

with tqdm(total=len(dois)) as pbar:
    with ThreadPoolExecutor(max_workers=N_THREADS) as executor:
        futures = [executor.submit(process_doi, doi, pbar) for doi in dois]
        for future in as_completed(futures):
            if future.exception():
                tqdm.write(f"  ✗ Unexpected error: {future.exception()}")

  0%|          | 7/6176 [00:00<03:32, 28.97it/s]

  ⏭ Skipping (already processed): 10.1163/9789004533257_004
  ⏭ Skipping (already processed): 10.47839/ijc.24.2.4012
  ⏭ Skipping (already processed): 10.1016/j.techsoc.2026.103342
  ⏭ Skipping (already processed): 10.1108/S1529-2134(2010)0000014011
  ⏭ Skipping (already processed): 10.1016/j.techsoc.2025.103121
  ⏭ Skipping (already processed): 10.4324/9781003687078-7
  ⏭ Skipping (already processed): 10.1007/s12232-026-00529-x
  ⏭ Skipping (already processed): 10.4324/9781003585428-6


  0%|          | 29/6176 [00:00<01:23, 73.44it/s]

  ⏭ Skipping (already processed): 10.1108/S1529-2134(2010)0000014009
  ⏭ Skipping (already processed): 10.1515/opth-2025-0042
  ⏭ Skipping (already processed): 10.1007/978-3-032-16145-1
  ⏭ Skipping (already processed): 10.1007/s12599-025-00925-7
  ⏭ Skipping (already processed): 10.4337/9781035344819
  ⏭ Skipping (already processed): 10.1007/s43621-025-02124-6
  ⏭ Skipping (already processed): 10.1093/9780191983627.003.0012
  ⏭ Skipping (already processed): 10.1007/978-981-96-0112-7_11
  ⏭ Skipping (already processed): 10.1007/978-981-96-0112-7_1
  ⏭ Skipping (already processed): 10.1007/s10639-025-13850-9
  ⏭ Skipping (already processed): 10.1080/08913811.2025.2489880
  ⏭ Skipping (already processed): 10.38178/07183089/1157240905
  ⏭ Skipping (already processed): 10.1215/00021482-11778200
  ⏭ Skipping (already processed): 10.1344/DIALECTOLOGIA.35.7
  ⏭ Skipping (already processed): 10.1002/soej.12738
  ⏭ Skipping (already processed): 10.4018/979-8-3693-8332-2.ch005
  ⏭ Skipping (alre

  ⏭ Skipping (already processed): 10.1002/soej.12747
  ⏭ Skipping (already processed): 10.31881/TLR.2026.019
  ⏭ Skipping (already processed): 10.1007/978-3-031-98494-5
  ⏭ Skipping (already processed): 10.1108/CG-03-2023-0093
  ⏭ Skipping (already processed): 10.1086/736914
  ⏭ Skipping (already processed): 10.1038/s41598-025-24608-1
  ⏭ Skipping (already processed): 10.4337/9781035350100
  ⏭ Skipping (already processed): 10.1111/ecaf.70005
  ⏭ Skipping (already processed): 10.1093/cje/beaf004
  ⏭ Skipping (already processed): 10.1108/JCM-11-2024-7371
  ⏭ Skipping (already processed): 10.1007/s11138-024-00648-0
  ⏭ Skipping (already processed): 10.1007/978-3-031-81002-2_6
  ⏭ Skipping (already processed): 10.1177/16094069251333894
  ⏭ Skipping (already processed): 10.1108/IJEBR-05-2025-0685
  ⏭ Skipping (already processed): 10.24507/icicelb.17.05.467
  ⏭ Skipping (already processed): 10.32518/sals1.2025.290
  ⏭ Skipping (already processed): 10.14422/pen.v81.i314.y2025.012
  ⏭ Skipping

...

  ⏭ Skipping (already processed): 10.1111/j.1536-7150.1997.tb03357.x
  ⏭ Skipping (already processed): 10.1017/S0007123497000197
  ⏭ Skipping (already processed): 10.1108/eb028708
  ⏭ Skipping (already processed): 10.1177/092137409200500302
  ⏭ Skipping (already processed): 10.1016/0304-3932(81)90007-6
  ⏭ Skipping (already processed): 10.1108/eb002671
  ⏭ Skipping (already processed): 10.1016/0883-9026(95)00109-3
  ⏭ Skipping (already processed): 10.1007/bf01103331
  ⏭ Skipping (already processed): 10.1177/053901886025001012
  ⏭ Skipping (already processed): 10.1111/j.1813-6982.1991.tb00966.x
  ⏭ Skipping (already processed): 10.1016/0167-2681(94)00028-D
  ⏭ Skipping (already processed): 10.1177/030981688803600106
  ⏭ Skipping (already processed): 10.1080/10370196.1997.11733243
  ⏭ Skipping (already processed): 10.1080/08913818908459563
  ⏭ Skipping (already processed): 10.1007/BF01102290
  ⏭ Skipping (already processed): 10.1007/BF02319876
  ⏭ Skipping (already processed): 10.1177/09

  ⏭ Skipping (already processed): 10.1080/09585199200000134
  ⏭ Skipping (already processed): 10.1016/0305-750X(94)90177-5
  ⏭ Skipping (already processed): 10.2202/1145-6396.1192
  ⏭ Skipping (already processed): 10.1016/S0305-750X(96)00117-9
  ⏭ Skipping (already processed): 10.1108/EUM0000000004861
  ⏭ Skipping (already processed): 10.1177/092137409200500305
  ⏭ Skipping (already processed): 10.1108/13552559610110727
  ⏭ Skipping (already processed): 10.1016/0959-8022(92)90007-F
  ⏭ Skipping (already processed): 10.1016/0954-349X(92)90026-3
  ⏭ Skipping (already processed): 10.1080/08913818908459574
  ⏭ Skipping (already processed): 10.1515/jeeh-1996-0410
  ⏭ Skipping (already processed): 10.1111/j.1467-6281.1987.tb00135.x
  ⏭ Skipping (already processed): 10.1108/EUM0000000000513
  ⏭ Skipping (already processed): 10.1016/0305-750X(90)90019-T
  ⏭ Skipping (already processed): 10.1080/01969728908902188
  ⏭ Skipping (already processed): 10.1017/S0143814X00001641
  ⏭ Skipping (already 

100%|█████████▉| 6175/6176 [00:47<00:00, 93.03it/s]

  ⏭ Skipping (already processed): 10.1007/BF01539298
  ⏭ Skipping (already processed): 10.1080/03585522.1992.10408250
  ⏭ Skipping (already processed): 10.1177/103530469200300201
  ⏭ Skipping (already processed): 10.1080/08913818908459578
  ⏭ Skipping (already processed): 10.1080/09668139608412344
  ⏭ Skipping (already processed): 10.1016/0922-1425(94)00040-Z
  ⏭ Skipping (already processed): 10.1007/bf03023450
  ⏭ Skipping (already processed): 10.1080/08913818808459544
  ⏭ Skipping (already processed): 10.1017/S0012217300012981
  ⏭ Skipping (already processed): 10.1002/mde.4090060314
  ⏭ Skipping (already processed): 10.2307/2129974
  ⏭ Skipping (already processed): 10.1111/j.1752-1688.1987.tb00830.x
  ⏭ Skipping (already processed): 10.1016/0277-3791(93)90048-Q


100%|██████████| 6176/6176 [00:47<00:00, 129.93it/s]


Segunda Onda de Coleta - Crawling

In [6]:
import pandas as pd

df = pd.read_csv('../data/processed/processed_dois.csv')
html_df = df[df['filetype'] == 'html']
html_df.head()

,doi,filepath,filetype
2,10.1007/978-3-031-97841-8_4,../data/processed/pdfs/10.1007_978-3-031-97841...,html
6,10.4324/9781003484745,../data/processed/pdfs/10.4324_9781003484745.html,html
7,10.1007/s11138-024-00657-z,../data/processed/pdfs/10.1007_s11138-024-0065...,html
9,10.1007/s11142-026-09937-4,../data/processed/pdfs/10.1007_s11142-026-0993...,html
11,10.1609/aaai.v40i35.40211,../data/processed/pdfs/10.1609_aaai.v40i35.402...,html


In [7]:
from bs4 import BeautifulSoup
from urllib.parse import urlparse, parse_qs, unquote


def extract_login_institution_link(html_path):
    with open(html_path, 'r', encoding='utf-8') as f:
        soup = BeautifulSoup(f, 'html.parser')
    
    a = soup.find('a',  string='log in via an institution')
    if a:
        url =  "http://" + a.get('href')
        redirect_uri = parse_qs(urlparse(url).query)["redirect_uri"][0]
        return redirect_uri
    
    return None

In [8]:
extract_login_institution_link("../data/processed/pdfs/10.1007_0-387-28181-9_27.html")

'https://link.springer.com/chapter/10.1007/0-387-28181-9_27'

In [ ]:
# https://sp.springer.com/saml/login?idp=https://cafe.ufmg.br/idp/shibboleth&targetUrl=https://link.springer.com/chapter/10.1007/0-387-28181-9_27

SyntaxError: invalid syntax (1372667014.py, line 1)

In [ ]:
"""
Login automático no portal Springer via SSO da UFMG usando Selenium,
seguido do download do PDF do artigo.

Dependências:
    pip install selenium webdriver-manager requests
"""

import os
import requests
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from webdriver_manager.chrome import ChromeDriverManager

# ── Credenciais ──────────────────────────────────────────────────────────────
USERNAME = "pcalais"
PASSWORD = "cruzeiro##1"

# ── Base SSO ─────────────────────────────────────────────────────────────────
SSO_BASE = (
    "https://sp.springer.com/saml/login"
    "?idp=https://cafe.ufmg.br/idp/shibboleth"
    "&targetUrl={target_url}"
)

# ── Timeout padrão (segundos) ─────────────────────────────────────────────────
TIMEOUT = 20


def build_driver(headless: bool = False) -> webdriver.Chrome:
    options = Options()
    if headless:
        options.add_argument("--headless=new")
    options.add_argument("--no-sandbox")
    options.add_argument("--disable-dev-shm-usage")
    options.add_argument("--disable-gpu")
    options.add_argument("--window-size=1280,900")
    options.add_argument(
        "user-agent=Mozilla/5.0 (X11; Linux x86_64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/124.0.0.0 Safari/537.36"
    )
    # Força o Chrome a baixar PDFs em vez de abrir no visualizador interno
    prefs = {
        "download.prompt_for_download": False,
        "plugins.always_open_pdf_externally": True,
    }
    options.add_experimental_option("prefs", prefs)

    service = Service(ChromeDriverManager().install())
    driver = webdriver.Chrome(service=service, options=options)
    return driver


def wait_for_field(driver, by, selector, timeout=TIMEOUT):
    return WebDriverWait(driver, timeout).until(
        EC.visibility_of_element_located((by, selector))
    )


def download_pdf_with_cookies(driver: webdriver.Chrome, pdf_url: str, dest_path: str):
    """
    Reutiliza os cookies da sessão autenticada do Selenium para baixar
    o PDF via requests (mais confiável do que clicar no link).
    """
    print(f"\n📥 Baixando PDF: {pdf_url}")

    # Exporta cookies do Selenium → requests
    session = requests.Session()
    for cookie in driver.get_cookies():
        session.cookies.set(cookie["name"], cookie["value"], domain=cookie.get("domain"))

    session.headers.update({
        "User-Agent": (
            "Mozilla/5.0 (X11; Linux x86_64) "
            "AppleWebKit/537.36 (KHTML, like Gecko) "
            "Chrome/124.0.0.0 Safari/537.36"
        ),
        "Referer": driver.current_url,
    })

    resp = session.get(pdf_url, stream=True, timeout=60)
    resp.raise_for_status()

    content_type = resp.headers.get("Content-Type", "")
    if "pdf" not in content_type and "octet-stream" not in content_type:
        print(f"   ⚠️  Content-Type inesperado: {content_type}")
        print("      O PDF pode exigir autenticação adicional ou estar bloqueado pela instituição.")

    with open(dest_path, "wb") as f:
        for chunk in resp.iter_content(chunk_size=8192):
            f.write(chunk)

    size_kb = os.path.getsize(dest_path) / 1024
    print(f"   ✅ PDF salvo em '{dest_path}' ({size_kb:.1f} KB)")


def login_and_download(target_url: str, dest_path: str = "artigo.pdf", headless: bool = False):
    """
    Autentica no SSO da UFMG e baixa o PDF do artigo Springer indicado.

    Parâmetros:
        target_url : URL do artigo no Springer
                     ex: "https://link.springer.com/article/10.1007/s11138-024-00657-z"
        dest_path  : caminho local onde o PDF será salvo (padrão: "artigo.pdf")
        headless   : True para rodar sem interface gráfica
    """
    start_url = SSO_BASE.format(target_url=target_url)
    driver = build_driver(headless=headless)

    try:
        # ── Passo 1: Abrir URL de entrada ─────────────────────────────────────
        print(f"[1/6] Abrindo: {start_url}")
        driver.get(start_url)

        # ── Passo 2: Aguardar formulário de login da UFMG ─────────────────────
        print("[2/6] Aguardando formulário de login da UFMG...")
        username_field = wait_for_field(driver, By.ID, "username")
        print(f"     → Página atual: {driver.current_url}")

        # ── Passo 3: Preencher credenciais ────────────────────────────────────
        print("[3/6] Preenchendo credenciais...")
        username_field.clear()
        username_field.send_keys(USERNAME)
        password_field = wait_for_field(driver, By.ID, "password")
        password_field.clear()
        password_field.send_keys(PASSWORD)

        # ── Passo 4: Submeter formulário ──────────────────────────────────────
        print("[4/6] Submetendo formulário...")
        try:
            submit_btn = driver.find_element(
                By.CSS_SELECTOR, "button[type='submit'], input[type='submit']"
            )
            submit_btn.click()
        except Exception:
            driver.execute_script("document.querySelector('form').submit();")

        # ── Passo 5: Aguardar chegada na página do artigo ─────────────────────
        print("[5/6] Aguardando redirecionamento para o Springer...")
        WebDriverWait(driver, TIMEOUT).until(
            lambda d: "link.springer.com" in d.current_url
        )
        print(f"     → URL final : {driver.current_url}")
        print(f"     → Título    : {driver.title}")

        # ── Passo 6: Localizar o link do PDF e baixar ─────────────────────────
        print("[6/6] Procurando link de download do PDF...")
        try:
            pdf_link_el = WebDriverWait(driver, TIMEOUT).until(
                EC.presence_of_element_located(
                    (By.CSS_SELECTOR, "a[data-article-pdf='true']")
                )
            )
            pdf_url = pdf_link_el.get_attribute("href")
            print(f"     → href do PDF: {pdf_url}")

        except Exception:
            # Fallback: constrói a URL padrão do Springer a partir do DOI
            current = driver.current_url
            doi_part = current.split("link.springer.com/article/")[-1].rstrip("/")
            pdf_url = f"https://link.springer.com/content/pdf/{doi_part}.pdf"
            print(f"     → Link não encontrado via DOM; URL construída: {pdf_url}")

        download_pdf_with_cookies(driver, pdf_url, dest_path)

    except Exception as exc:
        print(f"\n❌ Erro: {exc}")
        print(f"   URL no momento do erro: {driver.current_url}")
        driver.save_screenshot("erro_login.png")
        print("   Screenshot salvo em 'erro_login.png'")
        raise

    finally:
        driver.quit()
        print("🔒 Browser fechado.")


if __name__ == "__main__":
    # Exemplo de uso direto
    login_and_download(
        target_url="https://link.springer.com/article/10.1007/s11138-024-00657-z",
        dest_path="artigo-teste.pdf",
        headless=True,
    )


: 

In [ ]:
import os
import threading
import pandas as pd
from tqdm import tqdm
from bs4 import BeautifulSoup
from concurrent.futures import ThreadPoolExecutor, as_completed

N_THREADS = 6
PROCESSED_CSV = "../data/processed/dois_processed_round_2.csv"

# Load or create the tracking CSV
if os.path.exists(PROCESSED_CSV):
    processed_df = pd.read_csv(PROCESSED_CSV, dtype=str)
else:
    processed_df = pd.DataFrame(columns=["doi", "filepath", "success"])
    processed_df.to_csv(PROCESSED_CSV, index=False)

processed_dois = set(processed_df["doi"].tolist())

csv_lock = threading.Lock()

def append_result(doi, filepath, success):
    row = pd.DataFrame([{"doi": doi, "filepath": filepath, "success": success}])
    with csv_lock:
        row.to_csv(PROCESSED_CSV, mode="a", header=False, index=False)

def process_row(row):
    doi = row["doi"]
    filepath = row["filepath"]
    pdf_filepath = f"../data/processed/pdfs/{doi_to_filename(doi)}.pdf"

    success = False
    try:
        link = extract_login_institution_link(filepath)
        print(doi, filepath, link)

        if link is not None:
            print("EXTRACT")
            login_and_download(
                target_url=link,
                dest_path=pdf_filepath,
                headless=True,
            )
            print("SUCCESS")
            success = True

    except UnicodeDecodeError:
        with open(filepath, 'r', encoding='latin-1') as f:
            soup = BeautifulSoup(f, 'html.parser')
        a = soup.find('a', string='log in via an institution')
        link = a['href'] if a else None
        print(doi, pdf_filepath, link)
        success = link is not None

    except Exception as e:
        print(type(e).__name__, e)

    append_result(doi, pdf_filepath, success)
    return doi

rows_to_process = [
    row for _, row in html_df.iterrows()
    if row["doi"] not in processed_dois
]

with ThreadPoolExecutor(max_workers=N_THREADS) as executor:
    futures = {executor.submit(process_row, row): row["doi"] for row in rows_to_process}

    with tqdm(total=len(futures)) as pbar:
        for future in as_completed(futures):
            doi = futures[future]
            try:
                future.result()
            except Exception as e:
                print(f"Unhandled error for DOI {doi}: {type(e).__name__} {e}")
            pbar.update(1)

[1/6] Abrindo: https://sp.springer.com/saml/login?idp=https://cafe.ufmg.br/idp/shibboleth&targetUrl=https://link.springer.com/article/10.1007/s11747-012-0316-3
[5/6] Aguardando redirecionamento para o Springer...
     → URL final : https://link.springer.com/article/10.1007/s10273-013-1563-8
     → Título    : Vor der Bundestagswahl: Argumente für Mindestlöhne überzeugen nicht | Wirtschaftsdienst | Springer Nature Link
[6/6] Procurando link de download do PDF...
[1/6] Abrindo: https://sp.springer.com/saml/login?idp=https://cafe.ufmg.br/idp/shibboleth&targetUrl=https://link.springer.com/book/10.1057/9781137334275
[2/6] Aguardando formulário de login da UFMG...
     → Página atual: https://cafe.ufmg.br/idp/profile/SAML2/POST/SSO?execution=e1s2
[3/6] Preenchendo credenciais...
[2/6] Aguardando formulário de login da UFMG...
     → Página atual: https://cafe.ufmg.br/idp/profile/SAML2/POST/SSO?execution=e1s2
[3/6] Preenchendo credenciais...
[4/6] Submetendo formulário...
     → href do PDF: 

 75%|███████▍  | 5362/7195 [7:46:06<14:29:15, 28.45s/it]

🔒 Browser fechado.
SUCCESS
10.1007/978-3-642-29503-4_16 ../data/processed/pdfs/10.1007_978-3-642-29503-4_16.html https://link.springer.com/chapter/10.1007/978-3-642-29503-4_16
EXTRACT
     → Link não encontrado via DOM; URL construída: https://link.springer.com/content/pdf/https://link.springer.com/book/10.1057/9781137334275.pdf

📥 Baixando PDF: https://link.springer.com/content/pdf/https://link.springer.com/book/10.1057/9781137334275.pdf


 75%|███████▍  | 5363/7195 [7:46:08<10:27:29, 20.55s/it]

🔒 Browser fechado.
SUCCESS
🔒 Browser fechado.
SUCCESS
10.1007/978-1-4302-5942-8 ../data/processed/pdfs/10.1007_978-1-4302-5942-8.html https://link.springer.com/book/10.1007/978-1-4302-5942-8
EXTRACT
10.1007/s11127-011-9847-2 ../data/processed/pdfs/10.1007_s11127-011-9847-2.html https://link.springer.com/article/10.1007/s11127-011-9847-2
EXTRACT

❌ Erro: 400 Client Error: Bad Request for url: https://link.springer.com/content/pdf/https://link.springer.com/book/10.1057/9781137334275.pdf
   URL no momento do erro: https://link.springer.com/book/10.1057/9781137334275
   Screenshot salvo em 'erro_login.png'


...

🔒 Browser fechado.
SUCCESS
🔒 Browser fechado.
SUCCESS


 76%|███████▌  | 5443/7195 [7:52:05<1:16:03,  2.60s/it]

10.1016/j.infoecopol.2013.05.001 ../data/processed/pdfs/10.1016_j.infoecopol.2013.05.001.html None
10.1016/j.jebo.2011.10.018 ../data/processed/pdfs/10.1016_j.jebo.2011.10.018.html None
10.1057/9781137314253 ../data/processed/pdfs/10.1057_9781137314253.html https://link.springer.com/book/10.1057/9781137314253
EXTRACT


 76%|███████▌  | 5445/7195 [7:52:06<54:33,  1.87s/it]  

🔒 Browser fechado.
SUCCESS
10.1007/s11138-012-0189-1 ../data/processed/pdfs/10.1007_s11138-012-0189-1.html https://link.springer.com/article/10.1007/s11138-012-0189-1
EXTRACT
[1/6] Abrindo: https://sp.springer.com/saml/login?idp=https://cafe.ufmg.br/idp/shibboleth&targetUrl=https://link.springer.com/book/10.1057/9781137314253
[1/6] Abrindo: https://sp.springer.com/saml/login?idp=https://cafe.ufmg.br/idp/shibboleth&targetUrl=https://link.springer.com/chapter/10.1007/978-94-007-4011-2_5
[2/6] Aguardando formulário de login da UFMG...
[2/6] Aguardando formulário de login da UFMG...
     → Página atual: https://cafe.ufmg.br/idp/profile/SAML2/POST/SSO?execution=e1s2
[3/6] Preenchendo credenciais...
     → Página atual: https://cafe.ufmg.br/idp/profile/SAML2/POST/SSO?execution=e1s2
[3/6] Preenchendo credenciais...
